Resample refined labels:

In [ ]:
import os
import shutil
from glob import glob
from pathlib import Path
import SimpleITK as sitk
from monai.transforms import RemoveSmallObjects
import numpy as np
# === Paths ===
label_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"
image_dir = "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr"
out_image_dir = "/data/colon_cancer/nnUNet_raw/Dataset101_CC/imagesTr"
out_label_dir = "/data/colon_cancer/nnUNet_raw/Dataset101_CC/labelsTr"

# === Ensure output folders exist ===
os.makedirs(out_image_dir, exist_ok=True)
os.makedirs(out_label_dir, exist_ok=True)

# === Function to resample label to match image ===
def resample_label_to_image(label_path, image_ref_path):
    """Resample label to match reference image (CT or input image)."""
    label_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(image_ref_path))
    label_resampled = sitk.Resample(
        label_sitk,
        ref_sitk,
        sitk.Transform(),
        sitk.sitkNearestNeighbor,
        0,
        label_sitk.GetPixelID()
    )
    return label_resampled  # return as SimpleITK image

# === Iterate through labels ===
label_files = sorted(glob(os.path.join(label_dir, "*.nii.gz")))
print(f"Found {len(label_files)} label files")

for label_path in label_files:
    label_name = os.path.basename(label_path)              # e.g. 25_0000.nii.gz
    uid = label_name.split("_")[0]                         # e.g. 25
    img_path = os.path.join(image_dir, label_name)

    if os.path.exists(img_path):
        # Copy the image (keep original name)
        new_img_name = f"{uid.zfill(3)}_0000.nii.gz"  # e.g. 025_0000.nii.gz
        shutil.copy2(img_path, os.path.join(out_image_dir, new_img_name))

        # Resample and save the label
        label_resampled = resample_label_to_image(label_path, img_path)
        # Convert to numpy array for MONAI transform
        label_np = sitk.GetArrayFromImage(label_resampled).astype(np.uint8)
        delete_small = RemoveSmallObjects(min_size=200, connectivity=2)
        # Apply DeleteSmallObjects
        label_cleaned_np = delete_small(label_np)

        # Convert back to SimpleITK and preserve header info
        label_cleaned_sitk = sitk.GetImageFromArray(label_cleaned_np)
        label_cleaned_sitk.CopyInformation(label_resampled)

        # Save cleaned label
        new_label_name = f"{uid.zfill(3)}.nii.gz"
        sitk.WriteImage(label_cleaned_sitk, os.path.join(out_label_dir, new_label_name))

        


        print(f"Copied & Resampled: Image={new_img_name}, Label={new_label_name}")
    else:
        print(f"No matching image found for label {label_name}")
   
   

print("Done!")


Check refined labels:

In [1]:
import os
import numpy as np

# === CONFIG ===
LABEL_DIR =  "/data/colon_cancer/Task101_Colon/raw_splitted/imagesTr/labels/final"   # change this to your folder path
EXTENSIONS = [".npy", ".nii", ".nii.gz"]  # supported formats
THRESHOLD = 20  # minimum number of labeled voxels to consider non-empty

try:
    import nibabel as nib
    has_nibabel = True
except ImportError:
    has_nibabel = False


def load_label(filepath):
    """Load a label file (supports .npy or .nii/.nii.gz)."""
    ext = os.path.splitext(filepath)[1]
    if ext == ".npy":
        return np.load(filepath)
    elif ext in [".nii", ".gz"] and has_nibabel:
        return nib.load(filepath).get_fdata()
    else:
        raise ValueError(f"Unsupported format or missing nibabel: {filepath}")


def check_labels():
    label_files = [
        os.path.join(LABEL_DIR, f)
        for f in os.listdir(LABEL_DIR)
        if any(f.endswith(ext) for ext in EXTENSIONS)
    ]

    if not label_files:
        print(f"No label files found in {LABEL_DIR}")
        return

    volumes = []

    print(f"Found {len(label_files)} label files. Computing volumes...\n")

    for path in label_files:
        try:
            label = load_label(path)
            volume = np.sum(label > 0)
            volumes.append((path, volume))
        except Exception as e:
            print(f"⚠️ Could not load {path}: {e}")

    # Sort by volume
    volumes.sort(key=lambda x: x[1])

    print("=== Label volumes (sorted) ===")
    for path, vol in volumes:
        print(f"{os.path.basename(path):40s}  -> {vol}")

    print("\n=== Labels with empty or tiny annotations ===")
    for path, vol in volumes:
        if vol == 0:
            print(f"🚫 EMPTY: {os.path.basename(path)}")
        elif vol < THRESHOLD:
            print(f"⚠️ Low volume ({vol} voxels): {os.path.basename(path)}")

check_labels() 


Found 291 label files. Computing volumes...

=== Label volumes (sorted) ===
752_0000.nii.gz                           -> 7
177_0000.nii.gz                           -> 159
660_0000.nii.gz                           -> 161
693_0000.nii.gz                           -> 481
310_0000.nii.gz                           -> 679
696_0000.nii.gz                           -> 886
187_0000.nii.gz                           -> 1299
796_0000.nii.gz                           -> 1500
346_0000.nii.gz                           -> 1529
153_0000.nii.gz                           -> 1714
25_0000.nii.gz                            -> 1721
747_0000.nii.gz                           -> 1814
329_0000.nii.gz                           -> 1848
558_0000.nii.gz                           -> 1859
503_0000.nii.gz                           -> 1917
363_0000.nii.gz                           -> 2049
829_0000.nii.gz                           -> 2050
75_0000.nii.gz                            -> 2136
513_0000.nii.gz                 

Prepare Decathlon data:

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ----------------------------
# Paths
# ----------------------------
images_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs")  
labels_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/labelsTs")  
output_csv = Path("/data/colon_cancer/Classifier/Decathlon/labels.csv") 
"""
for img_path in images_folder.glob("*0000.nii.gz"): 
    # Extract number from filename
    number = img_path.name.replace("_0000.nii.gz", "")
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"._colon_{number_int}.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
"""


# ----------------------------
# Rename images
# ----------------------------
renamed_images = []

for img_path in images_folder.glob("colon_*.nii.gz"):
    # Extract number from filename
    number = img_path.name.replace("colon_", "").replace(".nii.gz", "")  
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"{number_int}_0000.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
    
    renamed_images.append(new_path)

# ----------------------------
# Rename labels
# ----------------------------
for lbl_path in labels_folder.glob("colon_*.nii.gz"):
    number = lbl_path.name.replace("colon_", "").replace(".nii.gz", "")  # '001' from 'colon_001'
    number_int = int(number)  # convert to int
    new_name = f"{number_int}.nii.gz"
    new_path = labels_folder / new_name
    shutil.move(str(lbl_path), str(new_path))

# ----------------------------
# Generate CSV
# ----------------------------
data = []

for img_path in sorted(images_folder.glob("*_0000.nii.gz")):
    uid = img_path.stem.split("_")[0]  # get the UID (number before '_0000')
    data.append({
        "UID": uid,
        "img_path": str(img_path),
        "target": 1,
        "Split": "test",
        "Fold": 0
    })

df = pd.DataFrame(data, columns=["UID", "img_path", "target", "Split", "Fold"])
df.to_csv(output_csv, index=False)
print(f"CSV saved to {output_csv}")


CSV saved to /data/colon_cancer/Classifier/Decathlon/labels.csv


In [18]:
from pathlib import Path
import shutil

# ----------------------------
# Paths
# ----------------------------
source_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs")  # Replace with your source folder
target_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra")  # Replace with your target folder
target_folder.mkdir(parents=True, exist_ok=True)  # Create target folder if it doesn't exist

# ----------------------------
# Move files
# ----------------------------
for file_path in source_folder.glob("._colon*"):
    shutil.move(str(file_path), str(target_folder / file_path.name))
    print(f"Moved {file_path.name} to {target_folder}")


Moved ._colon_92.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_102.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_159.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_38.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_181.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_111.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_88.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_139.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_50.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_75.nii.gz to /data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs_extra
Moved ._colon_193.nii.gz to /data/colon_cance